# 01 Agroforestry: planted-silvoarable first pass, Haiti

Computes CIAT's **partial terrain-and-soil index** (slope 50%, soil organic carbon 25%, pH 25%, with a host gate)
over the AOI, runs the **ordinal simulation** next to it, and compares both on maps.

Edit **only the parameters cell**, then run all. Every run saves its configuration, fingerprint, maps and a
manifest to `runs/<fingerprint>/`.

**Two ways to run it**

| | Quick overview (default) | Detailed |
|---|---|---|
| `RESOLUTION_M` | 5000 | 250 |
| `HOST_GATE` | `"skip"` | `"apply"` |
| Earth Engine requests | about 50 | about 500 |
| Meaning | slope, SOC and pH scores over Haiti, and how much the class numbers matter | the policy's gate included |

Each grid has its own cache folder, so changing the resolution never reuses another grid's layers.

- `MODE = "exploratory"`: the legal-exclusion mask is **not applied** and open decisions stay open, so the maps can be
  compared. Every map and the manifest say so. Not policy-conformant.
- `MODE = "policy"`: refuses to run until every decision is recorded.
- `DATA_SOURCE = "synthetic"` tests the pipeline offline; its results are never Haiti.

In [ ]:
# ===== PARAMETERS: edit only this cell =====
CONFIG_PATH = "config/runs/agroforestry_planted_first_pass.toml"   # base settings, relative to the repo root
MODE = "exploratory"                      # "exploratory" or "policy"
DATA_SOURCE = "gee"                       # "gee" or "synthetic" (offline pipeline test)
EE_PROJECT = ""                           # your Earth Engine project id (not committed anywhere)
GEE_TILE_INPUT_PIXELS = 2_500_000        # fine pixels one Earth Engine request may read; lower it if you see "User memory limit exceeded"
GEE_WORKERS = 4                          # Earth Engine requests in parallel (1 = one at a time)
USE_CACHE = True                          # read/save extracted layers in cache/ (extract from Earth Engine once)
REFRESH_CACHE = False                     # True: ignore the cache and extract again
CACHE_DIR = "cache/layers"
HOST_GATE = "skip"                        # "skip": WorldCover not read, every cell counts as host (fast, exploratory) | "apply": CIAT's gate (slow)
ONLY_FROM_CACHE = False                   # True: never call Earth Engine; use only what is already in the cache
OVERVIEW_M = 5000                         # block size of the coarse overview (a whole multiple of RESOLUTION_M)
MIN_SCORED_SHARE = 0.5                    # an overview block needs at least this share of its cells scored
SMALL_CHECK = False                       # look at a small box first (layers, ranges, maps) before the full AOI
SMALL_BBOX = (-72.45, 18.45, -72.25, 18.65)   # a small box near Port-au-Prince
SMALL_RESOLUTION_M = 250

# AOI (area of interest)
AOI_NAME = "Haiti (approximate box)"
AOI_BBOX = (-74.5, 18.0, -71.6, 20.1)     # minx, miny, maxx, maxy in EPSG:4326
AOI_SOURCE = ""                           # official boundary and where it comes from, for example "GADM 4.1"

# Grid
RESOLUTION_M = 5000                       # 5000 = quick overview (47 x 65 cells for Haiti); 250 = SoilGrids-native grid (much slower to extract)
SCENARIO = "baseline"

# Target and decisions (leave empty what is not decided)
ADMIN_COUNTRY = "HTI"
APPLICABILITY_DECISION = ""               # reference for using the India-derived curves in the target
SOIL_ADOPTED = False                      # True only with a decision reference below
SOIL_DECISION_REFERENCE = ""
LEGAL_MASK_SPEC = ""                      # assessed legal-exclusion rule and its coverage

# Ordinal simulation
N_DRAWS = 5000
SEED = 20261103                           # same seed, same result
SCALE_MODE = "independent"                # "independent" or "shared"
WEIGHT_CONCENTRATION = 0                  # 0 keeps CIAT's weights fixed; for example 20 lets them vary

# Where a layer is looked for, in this order (the first pass reads Earth Engine only, for now)
DATA_ORDER = ("local", "hf", "gee")
HF_REPO = "dagudelo30/nbs-ruralscan-data"

## 0. Environment

Finds the repository root (so the notebook works from any folder) and checks that the packages it needs are
available in **this Python**. If `nbs_runtime` is not installed it is loaded from `src/` instead. To install it
properly, run once from the repo root in the same environment as this kernel, then restart the kernel:
`python -m pip install -e ".[gee,viz]"`.

In [ ]:
import importlib.util
import sys
from pathlib import Path


def find_repo_root():
    here = Path.cwd().resolve()
    for p in (here, *here.parents):
        if (p / "src" / "nbs_runtime").is_dir() and (p / "pyproject.toml").exists():
            return p
    raise RuntimeError(f"Could not find the nbs-ruralscan-runtime repository above {here}. "
                       "Open this notebook from inside the repository folder.")


REPO_ROOT = find_repo_root()
print("python:", sys.executable)
print("repository:", REPO_ROOT)

if importlib.util.find_spec("nbs_runtime") is None:
    sys.path.insert(0, str(REPO_ROOT / "src"))
    print("nbs_runtime is not installed in this Python: loading it from", REPO_ROOT / "src")

needed = {"numpy": "numpy", "matplotlib": "matplotlib"}
if DATA_SOURCE == "gee":
    needed |= {"ee": "earthengine-api", "xee": "xee", "xarray": "xarray"}
missing = [pip_name for mod, pip_name in needed.items() if importlib.util.find_spec(mod) is None]
if missing:
    raise ModuleNotFoundError("Missing in this Python: " + ", ".join(missing) +
                              f"\nInstall with:  {sys.executable} -m pip install " + " ".join(missing) +
                              "\nthen restart the kernel.")
print("environment OK")

## 1. Run configuration

In [ ]:
from nbs_runtime.runconfig import RunConfig

OVERRIDES = {
    "run": {"mode": MODE},
    "aoi": {"name": AOI_NAME, "bbox": list(AOI_BBOX), "source": AOI_SOURCE},
    "grid": {"resolution_m": RESOLUTION_M, "scenario": SCENARIO},
    "target": {"country": ADMIN_COUNTRY, "applicability_decision": APPLICABILITY_DECISION},
    "soil": {"adopted": SOIL_ADOPTED, "decision_reference": SOIL_DECISION_REFERENCE},
    "masks": {"legal_mask_spec": LEGAL_MASK_SPEC},
    "simulation": {"n_draws": N_DRAWS, "seed": SEED, "scale_mode": SCALE_MODE,
                   "weight_concentration": WEIGHT_CONCENTRATION},
    "data": {"order": list(DATA_ORDER), "hf_repo": HF_REPO},
}
cfg = RunConfig.load(REPO_ROOT / CONFIG_PATH, OVERRIDES)

OUT_DIR = REPO_ROOT / "runs" / cfg.fingerprint()[:10]
OUT_DIR.mkdir(parents=True, exist_ok=True)
(OUT_DIR / "run_config.json").write_text(cfg.to_json(), encoding="utf-8")
print("run folder:", OUT_DIR.relative_to(REPO_ROOT), "| config fingerprint:", cfg.fingerprint()[:12])

## 2. Plan: what is ready and what is on hold

In [ ]:
from nbs_runtime.plan import plan

p = plan(cfg)
aoi = cfg.aoi
print(f"CIAT package {p.package_version} @ {p.package_ref[:8]} | policy {p.policy_id} | mode {cfg.run.mode}")
print(f"AOI: {aoi.name} | box area {aoi.area_km2():,.0f} km2 | "
      f"about {aoi.estimated_cells(cfg.grid.resolution_m):,} cells at {cfg.grid.resolution_m} m")
print("READY" if p.ready else "ON HOLD")
for h in p.holds:
    print("  hold:", h)
for w in p.warnings:
    print("  note:", w)

if cfg.run.mode == "policy" and not p.ready:
    raise SystemExit("policy mode needs every decision recorded; resolve the holds above, or use MODE = 'exploratory'")

## 3. Layers

Earth Engine, following CIAT's `DATASET_INPUTS.md`, on a **250 m grid** (SoilGrids is native 250 m, so soil, half of the
index weight, cannot support anything finer). The finer layers are aggregated, not resampled: slope is derived from
SRTM at its native ~30 m scale and **then** averaged to the grid; the 10 m WorldCover map becomes the share of each
class in every cell; soil is converted once and weighted by layer thickness (`planted_topsoil_0_30_v1`, a scoping
proposal CIAT has not selected). Missing values stay unknown: nothing is filled.

**The first extraction takes a while** (hundreds of requests, because Earth Engine reads every fine pixel). Each layer is
saved in `cache/layers/` as soon as it arrives, so an interrupted run resumes, and later runs, or a new CIAT package
version, never call Earth Engine again. Set `REFRESH_CACHE = True` to extract again.

In [ ]:
import numpy as np

from nbs_runtime.contract import load_package
from nbs_runtime.data import gee
from nbs_runtime.data.cache import LayerCache
from nbs_runtime.data.synthetic import synthetic_layers
from nbs_runtime.grid import Grid
from nbs_runtime.rasters import gate_class_groups
from nbs_runtime.viz import plot_layers

pkg = load_package()
policy = pkg.policy(cfg.run.policy_id)
groups = gate_class_groups(pkg.gate_params(policy))
print("host class groups from CIAT's gate:", groups)

if DATA_SOURCE == "gee":
    gee.init_ee(EE_PROJECT)


def load(grid):
    """Layers on a grid: Earth Engine, or synthetic data to test the pipeline offline."""
    if DATA_SOURCE == "synthetic":
        return synthetic_layers(grid, seed=cfg.simulation.seed)
    cache = LayerCache(REPO_ROOT / CACHE_DIR, grid, gee.ASSET_IDS, refresh=REFRESH_CACHE) if USE_CACHE else None
    if cache is not None:
        print("layer cache:", cache.dir.relative_to(REPO_ROOT))
    return gee.load_layers(grid, groups, cache=cache, max_input_pixels=GEE_TILE_INPUT_PIXELS,
                           host=HOST_GATE, cache_only=ONLY_FROM_CACHE, workers=GEE_WORKERS)


def describe(layers):
    for n in layers.notes:
        print("note:", n)
    for name, a in (("slope (degrees)", layers.slope_deg), ("SOC 0-30 cm (g/kg)", layers.soc_g_kg),
                    ("pH 0-30 cm", layers.ph), ("eligible host share", layers.eligible_fraction),
                    ("unknown host share", layers.unknown_fraction)):
        ok = a[~np.isnan(a)]
        rng = f"{ok.min():.3g} to {ok.max():.3g}, median {np.median(ok):.3g}" if ok.size else "no data"
        print(f"  {name:<22} {rng} | unknown {100 * np.isnan(a).mean():.1f}%")

### 3a. Quick check on a small box (optional)

Pulls the same layers for a small area and shows each one, so you can see them before the full run.
What to look for: slope in **degrees** (0 to a few tens), soil organic carbon in **g/kg** after conversion
(a few to tens, not hundreds and not below 1), pH between **4 and 9**, shares between 0 and 1. A map that is all
grey means nothing came back. Set `SMALL_CHECK = False` to skip it.

In [ ]:
if SMALL_CHECK:
    small = Grid.from_bbox(SMALL_BBOX, SMALL_RESOLUTION_M)
    print("small grid:", small.shape, "rows x columns at", SMALL_RESOLUTION_M, "m")
    small_layers = load(small)
    describe(small_layers)
    plot_layers(small_layers, small, "Quick check on a small box", OUT_DIR / "layers_small_check.png")
else:
    print("small check skipped")

### 3b. Layers for the AOI

The first time this cell runs it extracts from Earth Engine and prints its progress; afterwards it reads the cache.

In [ ]:
grid = Grid.from_aoi(cfg.aoi, cfg.grid.resolution_m)
print("grid:", grid.shape, "rows x columns at", cfg.grid.resolution_m, "m")
if DATA_SOURCE == "gee":
    n_requests = gee.request_plan(grid, HOST_GATE, GEE_TILE_INPUT_PIXELS)
    print("Earth Engine requests this run can make (layers already cached are skipped):", n_requests)
layers = load(grid)
describe(layers)
plot_layers(layers, grid, f"Input layers: {cfg.aoi.name}", OUT_DIR / "layers.png");

## 4. CIAT's calculation

Scores from CIAT's functions, host gate, weighted sum 50/25/25, unknown propagated, no renormalisation.

In [ ]:
from nbs_runtime.firstpass import run_first_pass
from nbs_runtime.simulate import build_level_space, simulate

space = build_level_space(pkg, policy)
fp = run_first_pass(space, layers, cfg.run.mode)

total = fp.status.size
print(f"cells: {total:,}")
for name, n in fp.status_counts().items():
    print(f"  {name:<38}{n:>8,}  {100 * n / total:5.1f}%")
if fp.scored.any():
    print(f"CIAT index over scored cells: min {np.nanmin(fp.pete):.3f}, median {np.nanmedian(fp.pete):.3f}, max {np.nanmax(fp.pete):.3f}")

## 5. Ordinal simulation

The order of the classes and the class of each cell stay fixed; the numbers given to the classes are redrawn many
times in increasing order. Each cell's index becomes a median and a 5 to 95% range, next to CIAT's own value.

In [ ]:
from nbs_runtime.compare import compare

s = cfg.simulation
res = simulate(space, s.n_draws, s.seed, s.scale_mode, s.prior, s.weight_concentration)
comp = compare(fp, res)

st = comp.stats
if st["n_scored"]:
    print(f"scored cells: {st['n_scored']:,} of {st['n_cells']:,}")
    print(f"rank agreement, CIAT vs simulation median (Spearman): {st['spearman_pete_vs_median']:.3f}")
    print(f"share of cells in CIAT's top third: {100 * st['share_pete_top_third']:.1f}%")
    print(f"agreement on top-third membership: {100 * st['top_third_agreement']:.1f}%")
    print(f"CIAT top-third cells that are fragile (chance of top third below 50%): {100 * st['share_of_pete_top_third_that_is_fragile']:.1f}%")
    print(f"cells with an uncertain top third (chance between 20% and 80%): {100 * st['share_uncertain_top_third']:.1f}%")
    print(f"width of the 5-95% range: median {st['width_median']:.2f}, 90th percentile {st['width_p90']:.2f}")
    print(f"cells where CIAT's value is above the simulation's 95th percentile: {100 * st['share_pete_above_p95']:.1f}%")
else:
    print("no scored cells: nothing to compare (policy mode, or no eligible host in the AOI)")

## 6. Maps

In [ ]:
import matplotlib.pyplot as plt

from nbs_runtime.viz import plot_comparison

flags = []
if cfg.run.mode == "exploratory":
    flags.append("EXPLORATORY: legal mask not applied, decisions open")
if layers.synthetic:
    flags.append("SYNTHETIC DATA: pipeline test, not Haiti")
if not layers.host_gate_applied:
    flags.append("HOST GATE NOT APPLIED: every cell counts as host")
if not cfg.aoi.source:
    flags.append("AOI is a placeholder box")
title = f"{'SYNTHETIC TEST: ' if layers.synthetic else ''}{cfg.aoi.name}: CIAT's index vs ordinal simulation ({cfg.simulation.scale_mode} scales, {cfg.simulation.n_draws:,} draws)"
note = " | ".join(flags) + f" | {cfg.grid.resolution_m} m grid | config {cfg.fingerprint()[:10]}"
fig = plot_comparison(fp, comp, grid, title, note, OUT_DIR / "maps.png")
plt.show()

## 7. Overview and sensitivity

The calculation stays at the fine grid; only the **result** is summarised in coarse blocks, so a block is never
scored from an averaged input. **How much the class numbers matter** is the point of this section:

- the overview maps show where the index is high and where its value depends most on the numbers given to the classes;
- the *ladder* shows, for every class combination present in the data, CIAT's index and the range the simulation gives it.
  Where two bars overlap, the order of those classes is not settled by the data: it depends on the numbers.

In [ ]:
from nbs_runtime.overview import make_overview
from nbs_runtime.viz import plot_combo_ladder, plot_overview

if OVERVIEW_M == cfg.grid.resolution_m:
    print(f"The grid is already the overview resolution ({OVERVIEW_M} m): the maps above are the overview.")
else:

    ov = make_overview(fp, comp, grid, OVERVIEW_M, MIN_SCORED_SHARE)
    print(f"overview: {ov.shape[0]} x {ov.shape[1]} blocks of {OVERVIEW_M} m "
          f"(a block needs at least {MIN_SCORED_SHARE:.0%} of its cells scored)")
    print(f"blocks shown: {int(np.isfinite(ov.ciat).sum()):,} of {ov.ciat.size:,}")
    plot_overview(ov, f"{'SYNTHETIC TEST: ' if layers.synthetic else ''}{cfg.aoi.name}: overview at {OVERVIEW_M / 1000:g} km",
                  note, OUT_DIR / "overview.png")
    plt.show()

In [ ]:
counts = fp.combo_counts(space.n_combos)
plot_combo_ladder(res, counts, f"{'SYNTHETIC TEST: ' if layers.synthetic else ''}Class combinations in the data: CIAT's index and the simulated range",
                  note, OUT_DIR / "class_ladder.png")
plt.show()

if fp.scored.any():
    p_scored = comp.p_top[fp.scored]
    print(f"scored cells: {int(fp.scored.sum()):,}. Chance of being in the top third:")
    for label, m in (("robustly in the top third (above 80%)", p_scored > 0.8),
                     ("uncertain (20% to 80%)", (p_scored >= 0.2) & (p_scored <= 0.8)),
                     ("robustly outside the top third (below 20%)", p_scored < 0.2)):
        print(f"  {label:<46}{100 * m.mean():5.1f}%")

## 8. Save this run

In [ ]:
import json


def f32(a):
    return None if a is None else np.asarray(a).astype("float32")


np.savez_compressed(
    OUT_DIR / "layers_and_results.npz",
    slope_deg=f32(layers.slope_deg), soc_g_kg=f32(layers.soc_g_kg), ph=f32(layers.ph),
    eligible_fraction=f32(layers.eligible_fraction), unknown_fraction=f32(layers.unknown_fraction),
    status=fp.status, ciat_index=f32(fp.pete), combo_id=fp.combo_id.astype("int16"),
    sim_median=f32(comp.median), sim_p05=f32(comp.p05), sim_p95=f32(comp.p95), p_top_third=f32(comp.p_top),
    depth_range=f32(fp.depth_range) if fp.depth_range is not None else np.array([], dtype="float32"),
    grid_bbox=np.array(grid.bbox), grid_resolution_deg=grid.resolution_deg,
)
manifest = {
    "config_fingerprint": cfg.fingerprint(),
    "aoi_fingerprint": cfg.aoi.fingerprint(),
    "mode": cfg.run.mode,
    "exploratory": cfg.run.mode == "exploratory",
    "data_source": DATA_SOURCE,
    "host_gate_applied": layers.host_gate_applied,
    "overview": {"overview_m": OVERVIEW_M, "min_scored_share": MIN_SCORED_SHARE},
    "synthetic": layers.synthetic,
    "ciat_package": {"version": pkg.version, "ref": pkg.pin["ref"], "source_commit": pkg.pin["package_source_commit"]},
    "plan": {"ready": p.ready, "holds": p.holds, "warnings": p.warnings},
    "layer_notes": layers.notes,
    "grid": {"shape": list(grid.shape), "resolution_m": grid.resolution_m, "bbox": list(grid.bbox)},
    "status_counts": fp.status_counts(),
    "comparison": {k: (None if isinstance(v, float) and np.isnan(v) else v) for k, v in comp.stats.items()},
    "simulation": res.metadata(),
}
(OUT_DIR / "manifest.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")
print("saved:", sorted(x.name for x in OUT_DIR.iterdir()))

## Reading the maps

- **CIAT index** and **simulation median** use the same 0-1 scale, but the simulation draws its scores with no
  prior information, so its median sits below CIAT's convention (0.25, 0.5, 0.75, 0.9). Compare the ranking, not the level.
- **5-95% width** shows where the result depends most on the numbers given to the classes.
- **Chance of the top third** is the share of scenarios in which the cell is among the best third of the scored cells.
- **Pixel state** keeps excluded, unknown and scored cells apart. Unknown is never turned into zero.

## Next

- Record the open decisions in the parameters cell as they are made (AOI boundary, applicability, soil, legal mask).
- Switch to `MODE = "policy"` once they are recorded.
- **Overview** blocks show the mean over the scored cells of a block; blocks with too few scored cells stay grey.
- **Host gate not applied** (`HOST_GATE = "skip"`) treats every cell as host. It is a fast, exploratory view of the three scores and of how much their numbers matter, not the policy's gate.
